In [1]:
from pathlib import Path
import duckdb

# 1. Khai báo đường dẫn
PROJECT_DIR = Path.cwd().parent
CSV_DIR = PROJECT_DIR / "data" / "ACME_Insurance" / "data"

# 2. Tạo thư mục lưu database
DB_DIR = PROJECT_DIR / "results" / "sql"
DB_DIR.mkdir(parents=True, exist_ok=True)

# 3. Kết nối DuckDB
DB_FILE = DB_DIR / "acme.duckdb"
conn = duckdb.connect(str(DB_FILE))

# 4. Kiểm tra
print("Database:", DB_FILE)
print("Connected:", True)
print("CSV files:", len(list(CSV_DIR.glob("*.csv"))))

Database: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\sql\acme.duckdb
Connected: True
CSV files: 29


In [2]:
# Lấy danh sách file CSV
csv_files = sorted(CSV_DIR.glob("*.csv"))

# Nạp từng file vào DuckDB
for file in csv_files:
    table_name = file.stem

    # Chuyển đường dẫn thành chuỗi dùng trong SQL
    csv_path = file.as_posix().replace("'", "''")

    # Tạo bảng từ file CSV
    conn.execute(f"""
        CREATE OR REPLACE TABLE "{table_name}" AS
        SELECT * FROM read_csv_auto(
            '{csv_path}',
            header = true,
            sample_size = -1
        )
    """)

    print(f"Loaded: {table_name}")

print("\nTotal tables loaded:", len(csv_files))

Loaded: Agreement
Loaded: Agreement_Assessment
Loaded: Agreement_Party_Role
Loaded: Assesment_Party_Role
Loaded: Assessment
Loaded: Assessment_Result
Loaded: Catastrophe
Loaded: Claim
Loaded: Claim_Amount
Loaded: Claim_Coverage
Loaded: Expense_Payment
Loaded: Expense_Reserve
Loaded: FireClaim
Loaded: Geographic_Location
Loaded: Insurable_Object
Loaded: Location_Address
Loaded: Loss_Payment
Loaded: Loss_Reserve
Loaded: Occurrence
Loaded: Organization
Loaded: Party
Loaded: Party_Role
Loaded: Person
Loaded: Physical_Location
Loaded: Policy
Loaded: Policy_Amount
Loaded: Policy_Coverage_Detail
Loaded: Premium
Loaded: Underwriting_Assessment

Total tables loaded: 29


In [3]:
import pandas as pd

results = []

for file in csv_files:
    table_name = file.stem

    # Số dòng trong CSV gốc
    csv_rows = len(pd.read_csv(file))

    # Số dòng trong DuckDB
    db_rows = conn.execute(
        f'SELECT COUNT(*) FROM "{table_name}"'
    ).fetchone()[0]

    results.append({
        "Table": table_name,
        "CSV Rows": csv_rows,
        "DB Rows": db_rows,
        "Match": csv_rows == db_rows
    })

# Hiển thị kết quả
validation_df = pd.DataFrame(results)
display(validation_df)

print("All tables match:", validation_df["Match"].all())

,Table,CSV Rows,DB Rows,Match
0,Agreement,2,2,True
1,Agreement_Assessment,2,2,True
2,Agreement_Party_Role,4,4,True
3,Assesment_Party_Role,2,2,True
4,Assessment,2,2,True
5,Assessment_Result,2,2,True
6,Catastrophe,4,4,True
7,Claim,2,2,True
8,Claim_Amount,8,8,True
9,Claim_Coverage,2,2,True


All tables match: True


In [4]:
# 13 bảng trong benchmark của tác giả
benchmark_tables = [
    "Claim", "Claim_Amount", "Loss_Payment", "Loss_Reserve",
    "Expense_Payment", "Expense_Reserve", "Claim_Coverage",
    "Policy_Coverage_Detail", "Policy", "Policy_Amount",
    "Agreement_Party_Role", "Premium", "Catastrophe"
]

# Lưu thông tin schema
schema_results = []

for table in benchmark_tables:
    columns = conn.execute(f'DESCRIBE "{table}"').df()

    for _, col in columns.iterrows():
        schema_results.append({
            "Table": table,
            "Column": col["column_name"],
            "DuckDB Type": col["column_type"]
        })

schema_df = pd.DataFrame(schema_results)

display(schema_df)
print("Tables checked:", schema_df["Table"].nunique())

,Table,Column,DuckDB Type
0,Claim,Claim_Identifier,BIGINT
1,Claim,Catastrophe_Identifier,BIGINT
2,Claim,Claim_Description,VARCHAR
3,Claim,Claims_Made_Date,VARCHAR
4,Claim,Company_Claim_Number,BIGINT
...,...,...,...
60,Catastrophe,Catastrophe_Identifier,BIGINT
61,Catastrophe,Catastrophe_Type_Code,VARCHAR
62,Catastrophe,Catastrophe_Name,VARCHAR
63,Catastrophe,Industry_Catastrophe_Code,VARCHAR


Tables checked: 13


In [5]:
# Các kiểu dữ liệu cần chuẩn hóa
type_fixes = {
    "Claim": {
        "Company_Claim_Number": "VARCHAR",
        "Claim_Open_Date": "TIMESTAMP",
        "Claim_Close_Date": "TIMESTAMP"
    },
    "Policy": {
        "Policy_Number": "VARCHAR",
        "Effective_Date": "TIMESTAMP",
        "Expiration_Date": "TIMESTAMP"
    },
    "Claim_Amount": {
        "Claim_Amount": "DECIMAL(15,2)"
    },
    "Policy_Amount": {
        "Policy_Amount": "DECIMAL(15,2)"
    }
}

# Thay đổi kiểu dữ liệu
for table, columns in type_fixes.items():
    for column, data_type in columns.items():
        conn.execute(
            f'ALTER TABLE "{table}" ALTER COLUMN "{column}" TYPE {data_type}'
        )

    print(f"Updated: {table}")

print("\nData types updated successfully!")

Updated: Claim
Updated: Policy
Updated: Claim_Amount
Updated: Policy_Amount

Data types updated successfully!


In [6]:
import re
import pandas as pd
from rdflib import Graph, Namespace, RDF

# 1. Đọc benchmark của tác giả
BENCHMARK_FILE = PROJECT_DIR / "data" / "ACME_Insurance" / "investigation" / "acme-benchmark.ttl"

benchmark = Graph()
benchmark.parse(BENCHMARK_FILE, format="turtle")

QA = Namespace("http://models.data.world/benchmarks/QandA#")
DWT = Namespace("https://templates.data.world/")

# 2. Chuẩn bị nơi lưu kết quả
run_log = []
gold_sql_results = {}
reference_sql = {}

# 3. Lấy tất cả câu hỏi
inquiries = list(benchmark.subjects(RDF.type, QA.Inquiry))

for inquiry in inquiries:
    question = str(benchmark.value(inquiry, QA.prompt))

    # Lấy các SQL Reference của câu hỏi
    refs = benchmark.objects(inquiry, QA.expects)

    for ref in refs:
        if (ref, RDF.type, DWT.SqlQuery) not in benchmark:
            continue

        ref_id = str(ref)
        sql = str(benchmark.value(ref, QA.queryText))

        # Lưu nguyên bản SQL
        reference_sql[ref_id] = sql

        # Bỏ dòng comment dùng ký hiệu # (DuckDB không hỗ trợ)
        sql_run = re.sub(r"(?m)^[ \t]*#.*$", "", sql).strip()

        # 4. Thực thi SQL
        try:
            if not re.match(r"(?i)^(SELECT|WITH)\b", sql_run):
                raise ValueError("Query is not SELECT/WITH")

            result = conn.execute(sql_run).df()

            gold_sql_results[ref_id] = result
            status = "OK"
            rows = len(result)
            error = ""

        except Exception as e:
            status = "ERROR"
            rows = None
            error = str(e).splitlines()[0]

        # 5. Ghi nhận kết quả
        run_log.append({
            "Question": question,
            "Reference ID": ref_id,
            "Status": status,
            "Rows": rows,
            "Error": error
        })

# 6. Tổng hợp
sql_log_df = pd.DataFrame(run_log)

print("Total SQL References:", len(sql_log_df))
print("\nExecution Status:")
print(sql_log_df["Status"].value_counts())

print("\nFailed Queries:")
display(sql_log_df[sql_log_df["Status"] == "ERROR"])

Total SQL References: 45

Execution Status:
Status
OK       44
ERROR     1
Name: count, dtype: int64

Failed Queries:


,Question,Reference ID,Status,Rows,Error
4,What is the average time to settle a claim by ...,https://templates.data.world/query-578baedd-93...,ERROR,NaN,"Binder Error: Referenced column ""day"" not foun..."


In [7]:
# 1. Lấy truy vấn bị lỗi
failed_id = sql_log_df.loc[
    sql_log_df["Status"] == "ERROR", "Reference ID"
].iloc[0]

original_sql = reference_sql[failed_id]

# 2. Chuyển cú pháp DATE_DIFF sang DuckDB
old = 'DATE_DIFF(claim_open_date, claim_close_date, "day")'
new = "date_diff('day', claim_open_date, claim_close_date)"

assert old in original_sql, "Unexpected SQL query"

fixed_sql = original_sql.replace(old, new)

# 3. Thực thi lại
result = conn.execute(fixed_sql).df()

gold_sql_results[failed_id] = result

sql_log_df.loc[
    sql_log_df["Reference ID"] == failed_id,
    ["Status", "Rows", "Error"]
] = ["OK", len(result), ""]

# 4. Lưu kết quả thực thi
OUTPUT_DIR = DB_DIR / "reference_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for ref_id, df in gold_sql_results.items():
    filename = ref_id.split("/")[-1] + ".csv"
    df.to_csv(OUTPUT_DIR / filename, index=False)

sql_log_df.to_csv(DB_DIR / "sql_execution_log.csv", index=False)

# 5. Tổng kết
print("Total SQL References:", len(sql_log_df))
print("Successful:", (sql_log_df["Status"] == "OK").sum())
print("Failed:", (sql_log_df["Status"] == "ERROR").sum())
print("Empty Results:", (sql_log_df["Rows"] == 0).sum())
print("Saved Results:", len(gold_sql_results))

Total SQL References: 45
Successful: 45
Failed: 0
Empty Results: 0
Saved Results: 45
